<a href="https://colab.research.google.com/github/HillaryDrugs/li7/blob/main/wav2vec2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


!pip -q install -U datasets transformers torchaudio jiwer accelerate torchcodec soundfile

import re
import torch
from datasets import load_dataset, Audio
from transformers import AutoProcessor, Wav2Vec2ForCTC
from jiwer import process_words

# -----------------------------
# 0) Settings
# -----------------------------
DATASET_NAME = "NightPrince/MasriSpeech-Full"

# ✅ FIXED model id (public + Arabic CTC)
MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

N_EVAL = 50  # change to 100/200 if you want

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------------
# 1) Load ORIGINAL dataset
# -----------------------------
ds = load_dataset(DATASET_NAME)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))  # requires torchcodec

print(ds)
print("Train:", len(ds["train"]), "| Validation:", len(ds["validation"]))

# -----------------------------
# 2) Load model + processor
# -----------------------------
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device)
model.eval()

# -----------------------------
# 3) Arabic normalization
# -----------------------------
arabic_diacritics = re.compile(r"[\u0617-\u061A\u064B-\u0652]")  # harakat
tatweel = "\u0640"

def normalize_arabic(text: str) -> str:
    if text is None:
        return ""
    text = str(text)

    # remove diacritics + tatweel
    text = re.sub(arabic_diacritics, "", text)
    text = text.replace(tatweel, "")

    # normalize common variants (helps scoring stability)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و").replace("ئ", "ي")

    # remove punctuation / non-arabic symbols
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

# -----------------------------
# 4) Transcribe function (CTC)
# -----------------------------
@torch.inference_mode()
def transcribe(audio_array):
    # audio_array already 16k because of cast_column(Audio(16000))
    inputs = processor(
        audio_array,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    input_values = inputs.input_values.to(device)

    logits = model(input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)

    hyp = processor.batch_decode(pred_ids)[0]
    return hyp

# -----------------------------
# 5) WER breakdown (your style)
# -----------------------------
def wer_breakdown(refs, hyps):
    """
    Perfect WER: 1 - exact match rate
    Deletion WER: D / N
    Typo WER: S / N   (typo ~ substitution)
    where N = total reference words
    """
    total_words = 0
    total_del = 0
    total_sub = 0
    perfect_count = 0

    for r, h in zip(refs, hyps):
        m = process_words(r, h)

        # N = hits + subs + dels
        n = m.hits + m.substitutions + m.deletions
        total_words += n
        total_del += m.deletions
        total_sub += m.substitutions

        if (m.substitutions + m.deletions + m.insertions) == 0:
            perfect_count += 1

    perfect_wer  = 1 - (perfect_count / len(refs)) if refs else 0.0
    deletion_wer = (total_del / total_words) if total_words else 0.0
    typo_wer     = (total_sub / total_words) if total_words else 0.0

    print(f"Perfect WER: {perfect_wer:.1f}")
    print(f"Deletion WER: {deletion_wer:.1f}")
    print(f"Typo WER: {typo_wer:.1f}")

# -----------------------------
# 6) Evaluate on validation subset
# -----------------------------
val = ds["validation"].select(range(min(N_EVAL, len(ds["validation"]))))

# Your dataset uses "transcription"
text_col = "transcription"
print("Using transcription column:", text_col)

refs, hyps = [], []

for i, ex in enumerate(val):
    audio = ex["audio"]
    ref_text = ex[text_col]

    hyp_text = transcribe(audio["array"])

    ref_n = normalize_arabic(ref_text)
    hyp_n = normalize_arabic(hyp_text)

    refs.append(ref_n)
    hyps.append(hyp_n)

    if i < 5:
        print("=" * 70)
        print("REF:", ref_text)
        print("HYP:", hyp_text)
        print("REF(norm):", ref_n)
        print("HYP(norm):", hyp_n)

# -----------------------------
# 7) Print metrics (your format)
# -----------------------------
wer_breakdown(refs, hyps)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Using transcription column: transcription
REF: شوفلنا المشوار ده يا حج
HYP: أوشبلاً ومشهار داحد
REF(norm): شوفلنا المشوار ده يا حج
HYP(norm): اوشبلا ومشهار داحد
REF: لأ للأسف دكتوره واحده بس بتعمل العمليه ديت عندنا في المحافظه
HYP: لا ألي لأسف ذكتور وحد بسبةن العملية لعة عمدلة المحضل
REF(norm): لا للاسف دكتوره واحده بس بتعمل العمليه ديت عندنا في المحافظه
HYP(norm): لا الي لاسف ذكتور وحد بسبةن العملية لعة عمدلة المحضل
REF: والراجل تبصله يعني إبن زمنه
HYP: والرادِل تبَصلُ يعْني أبن زمن
REF(norm): والراجل تبصله يعني ابن زمنه
HYP(norm): والرادل تبصل يعني ابن زمن
REF: و أنت كيف عرفته أبترل يا عمي
HYP: أُنْتَكِفِ اعْرِفْتْ وَبَتْرًلْ لِعَمِّي
REF(norm): و انت كيف عرفته ابترل يا عمي
HYP(norm): انتكف اعرفت وبترل لعمي
REF: ميعرفوش حاجه عن السوبر أه غير إنه لب
HYP: أَمَا عَرفوز حال جعَام الشوبَر- ه جالُ النُّلَد
REF(norm): ميعرفوش حاجه عن السوبر اه غير انه لب
HYP(norm): اما عرفوز حال جعام الشوبر ه جال النلد
Perfect WER: 1.0
Deletion WER: 0.2
Typo WER: 0.7
